# KYT Engine — EDA Analysis
Exploratory Data Analysis: классы, фичи, распределения

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

ROOT = Path().resolve().parent
DATA_DIR = ROOT / 'data' / 'raw'

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Загрузка данных

In [ ]:
nodes = pd.read_csv(DATA_DIR / 'nodes.csv')
classes = pd.read_csv(DATA_DIR / 'classes.csv')

print(f'nodes: {nodes.shape}')
print(f'classes: {classes.shape}')
print(f'Columns nodes: {list(nodes.columns)}')
print(f'Columns classes: {list(classes.columns)}')
nodes.head()

In [ ]:
classes.head()

In [ ]:
df = nodes.merge(classes, on='txId', how='inner')
df['label'] = df['label'].map({'illicit': 1, 'licit': 0}).fillna(0).astype(int)
print(f'Merged: {df.shape}')
df.head()

## 2. Распределение классов (illicit/licit)

In [ ]:
vc = df['label'].value_counts()
labels_map = {0: 'licit', 1: 'illicit'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

vc.plot.bar(ax=axes[0], color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[0].set_xticklabels([labels_map[i] for i in vc.index], rotation=0)
axes[0].set_title('Class Distribution (count)')
axes[0].set_ylabel('Count')

vc.plot.pie(ax=axes[1], autopct='%1.2f%%', colors=['#2ecc71', '#e74c3c'],
            labels=[labels_map[i] for i in vc.index], startangle=90)
axes[1].set_title('Class Distribution (%)')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print(f'Total: {len(df)}')
print(f'Illicit: {vc.get(1, 0)} ({vc.get(1, 0)/len(df)*100:.2f}%)')
print(f'Licit:   {vc.get(0, 0)} ({vc.get(0, 0)/len(df)*100:.2f}%)')

## 3. Статистики по числовым фичам

In [ ]:
num_cols = ['value', 'gas_price', 'gas_used', 'block_number', 'hour', 'day_of_week']
df[num_cols].describe().round(2)

In [ ]:
df[num_cols].info()

## 4. Feature Engineering для корреляционного анализа

In [ ]:
import sys
sys.path.insert(0, str(ROOT / 'src'))

from kyt_engine.features.engine import FeatureEngineer

fe = FeatureEngineer()
X = fe.fit_transform(df)
X['label'] = df.set_index('txId').loc[X.index, 'label'].values if 'txId' in df.columns else df['label'].values

print(f'Features shape: {X.shape}')
print(f'Feature count: {fe.n_features}')

## 5. Корреляционная матрица топ-20 фичей

In [ ]:
feat_cols = [c for c in X.columns if c != 'label']
label_corr = X[feat_cols].corrwith(X['label']).abs().sort_values(ascending=False)

top20 = label_corr.head(20).index.tolist()
print('Top-20 features by correlation with label:')
for i, (feat, corr) in enumerate(label_corr.head(20).items(), 1):
    print(f'  {i:2d}. {feat}: {corr:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 12))
corr_matrix = X[top20 + ['label']].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5,
            annot_kws={'size': 8}, ax=ax)
ax.set_title('Correlation Matrix — Top 20 Features + Label', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Визуализация распределений illicit vs licit

In [ ]:
top8 = label_corr.head(8).index.tolist()

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, feat in enumerate(top8):
    ax = axes[i]
   licit = X.loc[X['label'] == 0, feat]
    illicit = X.loc[X['label'] == 1, feat]

    ax.hist(licit, bins=50, alpha=0.6, label='licit', color='#2ecc71', density=True)
    ax.hist(illicit, bins=50, alpha=0.6, label='illicit', color='#e74c3c', density=True)
    ax.set_title(f'{feat}\n(r={label_corr[feat]:.3f})', fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions: Illicit vs Licit (Top 8)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, feat in enumerate(top8):
    ax = axes[i]
    data = X[[feat, 'label']].copy()
    data['class'] = data['label'].map({0: 'licit', 1: 'illicit'})
    sns.boxplot(data=data, x='class', y=feat, palette=['#2ecc71', '#e74c3c'], ax=ax)
    ax.set_title(feat, fontsize=10)

plt.suptitle('Boxplots: Illicit vs Licit (Top 8 Features)', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Распределения raw-признаков

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    ax = axes[i]
    for label_val, color, name in [(0, '#2ecc71', 'licit'), (1, '#e74c3c', 'illicit')]:
        subset = df.loc[df['label'] == label_val, col]
        ax.hist(subset, bins=50, alpha=0.6, color=color, label=name, density=True)
    ax.set_title(col)
    ax.legend()

plt.suptitle('Raw Feature Distributions by Class', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(['value', 'gas_price', 'gas_used']):
    ax = axes[i]
    for label_val, color, name in [(0, '#2ecc71', 'licit'), (1, '#e74c3c', 'illicit')]:
        subset = df.loc[df['label'] == label_val, col]
        ax.hist(np.log1p(subset), bins=50, alpha=0.6, color=color, label=name, density=True)
    ax.set_title(f'log(1 + {col})')
    ax.legend()

plt.suptitle('Log-transformed Distributions by Class', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Temporal patterns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for label_val, color, name in [(0, '#2ecc71', 'licit'), (1, '#e74c3c', 'illicit')]:
    subset = df[df['label'] == label_val]
    subset['hour'].value_counts().sort_index().plot(
        ax=axes[0], color=color, label=name, alpha=0.7
    )
axes[0].set_title('Transactions by Hour of Day')
axes[0].set_xlabel('Hour')
axes[0].legend()

for label_val, color, name in [(0, '#2ecc71', 'licit'), (1, '#e74c3c', 'illicit')]:
    subset = df[df['label'] == label_val]
    subset['day_of_week'].value_counts().sort_index().plot(
        ax=axes[1], color=color, label=name, alpha=0.7
    )
axes[1].set_title('Transactions by Day of Week')
axes[1].set_xlabel('Day (0=Mon, 6=Sun)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Выводы по данным

In [ ]:
print('=== DATA SUMMARY ===')
print(f'Total transactions: {len(df)}')
print(f'Unique addresses: {df["address"].nunique()}')
print(f'Illicit ratio: {df["label"].mean():.4f} ({df["label"].sum()}/{len(df)})')
print()
print('=== KEY FINDINGS ===')
print(f'1. Dataset is imbalanced: {df["label"].value_counts().to_dict()}')
print(f'2. Top predictive features (corr with label):')
for feat, corr in label_corr.head(5).items():
    print(f'   - {feat}: {corr:.4f}')
print(f'3. Feature engineering produced {fe.n_features} features from {df.shape[1]} raw columns')
print(f'4. Behavioral features add temporal/strategic signal beyond raw transaction data')
print()
print('=== RECOMMENDATIONS ===')
print('- Use class_weight="balanced" or SMOTE for training')
print('- LightGBM + Autoencoder ensemble should capture both tabular and anomaly signals')
print('- Stratified split is critical given class imbalance')